# 01 · Big-O and timing

After this notebook you can state the time and space complexity of the code you write in an interview, back the claim up with a two-minute timing experiment, and say what every common Python built-in costs.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** basic Python ([03 · Lists and tuples](../01_python_basics/03_lists_and_tuples.ipynb)) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 1 "Big-O & complexity"

## Why this matters in interviews

- Every coding round ends with *"What is the time and space complexity?"* An answer without a reason ("O(n), because each element is pushed and popped at most once") loses points.
- The jump from brute force to optimal is almost always a complexity argument: a nested loop becomes a hash-map lookup, a scan becomes a binary search.
- The usual follow-ups: *"Can you do better?"*, *"Is dict lookup really O(1)?"*, *"What does amortised mean?"*, *"What does `x in my_list` cost?"*, *"Why is `list.pop(0)` slow?"*
- GenAI interviews use the same reasoning: attention is O(n²) in context length, exact vector search is O(N·d) per query, KV-cache memory grows linearly with context and batch.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tr11` | Why is attention O(n²), and what have people done about it? |
| `ve03` | Your vector database has 10 million documents. How does it find the nearest without comparing all 10 million? |
| `in02` | Explain the KV cache — what it stores, why it works, and what it costs. |

In [ ]:
import bisect
import heapq
import math
import random
import sys
import time
import tracemalloc
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

random.seed(0)
rng = np.random.default_rng(0)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})


def best_time(fn, number=1, repeat=5):
    """Seconds per call of fn(): the fastest of `repeat` runs of `number` calls.
    The minimum filters out noise from other programs; it is the standard way to time small code."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        best = min(best, (time.perf_counter() - t0) / number)
    return best


def interleaved_times(calls, number=1, rounds=5):
    """best_time for several calls, timed in alternating rounds so background load hits them all equally."""
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            best[i] = min(best[i], best_time(call, number=number, repeat=1))
    return best


print(f"Python {sys.version.split()[0]}, numpy {np.__version__}")

## 1. What Big-O measures

**In plain English:** Big-O answers one question — *if the input gets bigger, how fast does the cost grow?* Not "how many seconds" but "what shape": double the input and an O(n) algorithm takes about twice as long, an O(n²) one four times as long, an O(log n) one barely longer.

The rules:

1. **Count steps as a function of the input size** and name every size (`n` elements, `k` window, `V` and `E` for graphs, `L` for string length).
2. **Keep the dominant term, drop constants:** `3n + 5` → O(n); `n²/2 + 10n` → O(n²). Constants matter in practice, but not to the growth rate.
3. **Worst case unless you say otherwise.** Hash lookups are O(1) *on average* and O(n) in the pathological worst case; quicksort is O(n log n) on average and O(n²) worst.
4. **Time and space are two separate answers.** Always give both.

Formally, *f(n) = O(g(n))* means f grows no faster than g, up to a constant factor, for large n (an upper bound); Θ is the tight bound and Ω the lower bound. In interviews people say "Big-O" and mean the tight bound: call a single loop O(n), not O(n²), even though O(n²) is technically also an upper bound.

Count the steps of three shapes of code and watch what happens when `n` grows 10×:

In [ ]:
def linear_steps(n):            # one pass over the input
    return sum(1 for _ in range(n))


def pair_steps(n):              # every pair i < j: n(n-1)/2 steps
    return sum(1 for i in range(n) for j in range(i + 1, n))


def halving_steps(n):           # halve the problem until one item is left
    steps = 0
    while n > 1:
        n //= 2
        steps += 1
    return steps


print(f"{'n':>6} {'one loop':>9} {'all pairs':>10} {'halving':>8}")
for n in [10, 100, 1000]:
    print(f"{n:>6} {linear_steps(n):>9} {pair_steps(n):>10} {halving_steps(n):>8}")

assert pair_steps(1000) == 1000 * 999 // 2      # n²/2 steps is still O(n²): the 1/2 is dropped
assert halving_steps(1024) == 10 == math.log2(1024)

Every 10× in `n`: the single loop does 10× the work, the pairs 100×, and the halving loop just +3 steps. That is O(n), O(n²) and O(log n).

## 2. The common classes

| Big-O | Name | Typical code | n = 1,000 | n = 1,000,000 |
|---|---|---|---|---|
| O(1) | constant | `d[key]`, `lst[i]`, `lst.append(x)` | 1 | 1 |
| O(log n) | logarithmic | binary search, `heapq.heappush` | ~10 | ~20 |
| O(n) | linear | one loop, `sum(lst)`, `x in lst` | 10³ | 10⁶ |
| O(n log n) | linearithmic | `sorted(lst)`, merge sort | ~10⁴ | ~2·10⁷ |
| O(n²) | quadratic | all pairs, nested loops, bubble sort | 10⁶ | 10¹² (hours) |
| O(2ⁿ) | exponential | all subsets, naive recursive Fibonacci | ~10³⁰¹ | — |
| O(n!) | factorial | all permutations | — | — |

A picture of how quickly they separate (note the log-scale y-axis — on a normal axis everything but the top two would be a flat line):

In [ ]:
n = np.arange(2, 65)
curves = {"O(1)": np.ones(len(n)), "O(log n)": np.log2(n), "O(n)": n, "O(n log n)": n * np.log2(n),
          "O(n²)": n ** 2.0, "O(2ⁿ)": 2.0 ** n}

fig, ax = plt.subplots(figsize=(9, 4.2))
for label, steps in curves.items():
    ax.plot(n, steps, label=label)
ax.set_yscale("log")
ax.set_ylim(0.8, 1e5)
ax.set(title="How the common complexity classes grow", xlabel="input size n", ylabel="steps (log scale)")
ax.legend(ncol=2)
plt.show()

## 3. Measure growth empirically

Theory says what *should* happen; a timing experiment checks it. The trick: time each operation at doubling sizes and plot on **log-log axes**. If time ∝ nᵏ then log t = k·log n + c, so **the slope of the line is the exponent k**: slope 0 → O(1), 1 → O(n), 2 → O(n²). O(log n) looks almost flat and O(n log n) slightly steeper than 1.

Five operations, one per class. The helpers below are plain Python so you can see exactly what is being counted.

In [ ]:
def binary_search(a, x):
    """Index of the first element >= x in the sorted list a (notebook 06 explains the template)."""
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


def linear_search(a, x):
    for i, v in enumerate(a):
        if v == x:
            return i
    return -1


def count_inversions_brute(a):
    """Pairs i < j with a[i] > a[j], by checking every pair: n(n-1)/2 comparisons."""
    count = 0
    for i in range(len(a)):
        for j in range(i + 1, len(a)):
            if a[i] > a[j]:
                count += 1
    return count

In [ ]:
def setup_dict(n):
    d = {i: i for i in range(n)}
    return lambda: d[n // 2]


def setup_binary(n):
    a = list(range(n))
    return lambda: binary_search(a, n // 3)


def setup_linear(n):
    a = list(range(n))
    return lambda: linear_search(a, -1)          # worst case: the value is not there


def setup_sort(n):
    a = [random.random() for _ in range(n)]
    return lambda: sorted(a)


def setup_pairs(n):
    a = [random.random() for _ in range(n)]
    return lambda: count_inversions_brute(a)


sizes = [2 ** k for k in range(8, 17)]           # 256 ... 65,536
plan = [("O(1): dict lookup", setup_dict, sizes, 20_000),
        ("O(log n): binary search", setup_binary, sizes, 2_000),
        ("O(n): linear search", setup_linear, sizes, 2),
        ("O(n log n): sorted()", setup_sort, sizes, 1),
        ("O(n²): check all pairs", setup_pairs, [2 ** k for k in range(7, 12)], 1)]   # 128 ... 2,048

t_start = time.perf_counter()
results = {label: (ns, interleaved_times([setup(n) for n in ns], number=number))
           for label, setup, ns, number in plan}
print(f"timed {sum(len(ns) for ns, _ in results.values())} (operation, size) points in {time.perf_counter() - t_start:.1f} s")

In [ ]:
slopes = {}
print(f"{'operation':26} {'fitted slope':>12} {'avg t(2n) / t(n)':>17}")
for label, (ns, ts) in results.items():
    slopes[label] = np.polyfit(np.log2(ns), np.log2(ts), 1)[0]
    print(f"{label:26} {slopes[label]:>12.2f} {2 ** slopes[label]:>17.2f}")   # slope k -> doubling multiplies time by 2^k

s_const, s_log, s_lin, s_nlogn, s_quad = slopes.values()
assert s_const < 0.5 and s_log < 0.5, "O(1) and O(log n) should be (nearly) flat"
assert 0.75 < s_lin < 1.3, "O(n): doubling n should double the time"
assert 0.85 < s_nlogn < 1.5, "O(n log n): a little steeper than linear"
assert s_quad > 1.6, "O(n²): doubling n should quadruple the time"

sort_vs_scan = results["O(n log n): sorted()"][1][-1] / results["O(n): linear search"][1][-1]
print(f"\nat n = 65,536: sorted() takes {sort_vs_scan:.1f}x as long as one pure-Python scan (16x the comparisons)")
assert sort_vs_scan < 16          # constants: C code beats a Python loop

fig, ax = plt.subplots(figsize=(11, 4.5))
for label, (ns, ts) in results.items():
    ax.loglog(ns, np.array(ts) * 1e6, marker="o", ms=4, label=f"{label}  (slope {slopes[label]:.2f})")
ax.set(title="Measured growth: on a log-log plot the slope is the exponent",
       xlabel="input size n (log scale)", ylabel="time per call, µs (log scale)")
ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

The last column is **the doubling test** — the one experiment worth remembering: double `n` and see what happens to the time (here averaged over all the doublings as 2^slope, because a single pair of timings is noisy on a busy machine). ≈1 → O(1) or O(log n); ≈2 → O(n); a bit over 2 → O(n log n); ≈4 → O(n²); ≈8 → O(n³). It works on any black box, including your own interview solution.

Note the *levels* too: at n = 65,536 sorting does about log₂ n = 16× more comparisons than one scan, yet the printed ratio shows the C-implemented `sorted()` is far less than 16× slower than the pure-Python scan. Big-O is about growth; constants (C vs Python, cache misses) decide who wins at a given n.

### What n can you afford?

Interview problems state constraints ("n ≤ 10⁵") precisely so you can work out the intended complexity. Measure how many simple Python steps this machine does per second, then find the largest n each class can handle in about one second:

In [ ]:
def simple_loop(n):
    total = 0
    for i in range(n):
        total += i
    return total


steps_per_second = 1_000_000 / best_time(lambda: simple_loop(1_000_000), repeat=3)


def largest_n(cost, budget):
    """Largest integer n with cost(n) <= budget, for an increasing cost function."""
    lo, hi = 1, 2
    while cost(hi) <= budget:            # double until it is too big ...
        lo, hi = hi, hi * 2
    while hi - lo > 1:                   # ... then binary-search the boundary (notebook 06)
        mid = (lo + hi) // 2
        lo, hi = (mid, hi) if cost(mid) <= budget else (lo, mid)
    return lo


costs = {"O(n)": lambda n: n, "O(n log n)": lambda n: n * math.log2(n), "O(n²)": lambda n: n ** 2,
         "O(n³)": lambda n: n ** 3, "O(2ⁿ)": lambda n: 2 ** n, "O(n!)": math.factorial}
print(f"~{steps_per_second / 1e6:.0f} million simple Python steps per second on this machine")
print("largest n that fits in about one second of pure Python:")
for name, cost in costs.items():
    print(f"  {name:11} n ≈ {largest_n(cost, steps_per_second):,}")

So in Python: n ≈ 10⁵–10⁶ means "find an O(n) or O(n log n) solution", a few thousand means O(n²) is acceptable, ~20 means exponential search over subsets is intended. (C++ does roughly 10–50× more steps per second, which is why published constraint tables look more generous.)

## 4. Amortised O(1): how `list.append` stays cheap

**In plain English:** a Python list is a *dynamic array* — one block of memory with spare slots at the end. `append` drops the item into the next spare slot: O(1). When the block is full, Python allocates a bigger block and copies every element across: O(n). Because the block grows by a *proportion* of its size, those expensive copies get rarer as the list grows, and the average cost per append is a constant. That is **amortised O(1)**: any sequence of n appends costs O(n) in total, even though an individual append is occasionally O(n).

`sys.getsizeof` shows the hidden capacity (each slot is an 8-byte pointer):

In [ ]:
EMPTY = sys.getsizeof([])        # bytes of an empty list object
PTR = 8                          # one pointer per slot on a 64-bit build

lst, lengths, capacities, resizes = [], [], [], []
for i in range(150):
    before = (sys.getsizeof(lst) - EMPTY) // PTR
    lst.append(i)
    after = (sys.getsizeof(lst) - EMPTY) // PTR
    lengths.append(len(lst))
    capacities.append(after)
    if after != before:
        resizes.append((len(lst), before, after))

print("append that caused a resize -> capacity before -> after")
for length, before, after in resizes:
    print(f"  append #{length:<4} {before:>4} -> {after:<4} (+{after - before})")

# CPython's rule (Objects/listobject.c): new capacity = (n + n//8 + 6) rounded down to a multiple of 4
for length, _, after in resizes:
    assert after == (length + (length >> 3) + 6) & ~3

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(lengths, capacities, where="post", label="capacity (slots allocated)")
ax.plot(lengths, lengths, label="length (slots used)")
ax.set(title="A list over-allocates: capacity jumps ahead of length by ~12.5%",
       xlabel="number of appends", ylabel="slots")
ax.legend()
plt.show()

Now the amortised argument in numbers. Assume the worst case — every resize copies every existing element — and count copies per append as the list grows:

In [ ]:
def copies_per_append(n):
    """Worst-case element copies caused by n appends, divided by n."""
    lst, capacity, copies = [], 0, 0
    for i in range(n):
        if len(lst) == capacity:         # full: this append moves everything to a bigger block
            copies += len(lst)
        lst.append(i)
        capacity = (sys.getsizeof(lst) - EMPTY) // PTR
    return copies / n


ratios = {n: copies_per_append(n) for n in [1_000, 10_000, 100_000]}
for n, r in ratios.items():
    print(f"n = {n:>7,} appends: {r:.1f} copies per append")
assert max(ratios.values()) < 10 and max(ratios.values()) / min(ratios.values()) < 1.3   # flat: O(1) amortised

A constant (~8, because the growth factor is ~1/8) regardless of n: amortised O(1). Growing by a *fixed* number of slots instead would make copies per append grow with n — exercise 3 below proves it.

## 5. Space complexity

**In plain English:** how much *extra* memory does the algorithm need as n grows? Say whether you count the input and the output (usually you don't; say "O(1) extra space, not counting the output").

| Extra space | Typical cause |
|---|---|
| O(1) | a few variables — two pointers, running sums |
| O(log n) | recursion depth of a balanced divide-and-conquer (quicksort on average) |
| O(n) | a copy, a hash set of seen items, a result list, recursion depth n |
| O(n·m) | a 2-D DP table |

`tracemalloc` (standard library) measures it. A list comprehension materialises all n values; a generator produces them one at a time:

In [ ]:
def peak_kib(fn):
    """Peak memory allocated by Python while fn() runs, in KiB."""
    tracemalloc.start()
    fn()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak / 1024


print(f"{'n':>8} {'sum([x*x ...]) list':>20} {'sum(x*x ...) generator':>23}")
for n in [1_000, 10_000, 100_000]:
    list_peak = peak_kib(lambda: sum([x * x for x in range(n)]))
    gen_peak = peak_kib(lambda: sum(x * x for x in range(n)))
    print(f"{n:>8,} {list_peak:>16,.0f} KiB {gen_peak:>19,.1f} KiB")
assert gen_peak < 0.01 * list_peak          # O(1) vs O(n) extra space

a = list(range(100_000))
print(f"\na[::-1]     (new list): {peak_kib(lambda: a[::-1]):,.0f} KiB extra")
print(f"a.reverse() (in place): {peak_kib(a.reverse):,.0f} KiB extra")

**Recursion uses space too.** Every pending call is a stack frame, so a recursion of depth n is O(n) space even if it stores nothing else — and Python caps the depth (default 1000, no tail-call optimisation):

In [ ]:
def sum_recursive(a, i=0):
    """Sum of a[i:], recursively: O(n) time and O(n) stack space."""
    if i == len(a):
        return 0
    return a[i] + sum_recursive(a, i + 1)


print("recursion limit:", sys.getrecursionlimit())
print("sum_recursive(range(500)) =", sum_recursive(list(range(500))))
try:
    sum_recursive(list(range(5_000)))
    blew_up = False
except RecursionError as err:
    blew_up = True
    print("sum_recursive(range(5000)) -> RecursionError:", err)
assert blew_up

That is why interview answers in Python prefer iterative DFS with an explicit stack for deep inputs (a 10⁵-node path graph, a degenerate tree).

## 6. What Python's built-ins cost

Average case, n = size of the container ([CPython's own table](https://wiki.python.org/moin/TimeComplexity) is the reference):

| Structure | O(1) | O(log n) | O(n) or worse |
|---|---|---|---|
| `list` (dynamic array) | `a[i]`, `a[i] = x`, `len(a)`, `a.append(x)` (amortised), `a.pop()` | — | `x in a`, `a.index(x)`, `a.insert(0, x)`, `a.pop(0)`, `a.remove(x)`, `del a[i]` (O(n−i)), slice `a[i:j]` (O(j−i)), `a + b`, `min/max/sum`; `a.sort()` O(n log n) |
| `dict` (hash table) | `d[k]`, `d[k] = v`, `k in d`, `del d[k]`, `d.get(k)` | — | iteration, `d.copy()`; **worst case** O(n) per op if keys collide |
| `set` (hash table) | `s.add(x)`, `x in s`, `s.discard(x)` | — | `s.union(t)` O(len s + len t), `s & t` O(min(len s, len t)), `s - t` O(len s) |
| `collections.deque` | `append`, `appendleft`, `pop`, `popleft`, `d[0]`, `d[-1]` | — | `d[i]` in the middle, `x in d`, `rotate(k)` O(k) |
| `heapq` on a list | `h[0]` (peek min) | `heappush`, `heappop`, `heapreplace` | `heapify` O(n), `nlargest(k, xs)` O(n log k) |
| `str` (immutable) | `len(s)`, `s[i]` | — | `s + t`, slicing, `sub in s`, `s.split()`, `s.replace()`, `"".join(parts)` O(total length), `s == t` |
| `sorted` / `list.sort` | — | — | O(n log n) worst, **O(n) on already-sorted input** (Timsort), O(n) extra space |
| `bisect` | — | `bisect_left`, `bisect_right` | `insort` is **O(n)** — the search is O(log n) but the insert shifts elements |

Don't take a table's word for it — measure the claims. Every operation below runs on containers of 10,000 and of 100,000 items; 10× the size should leave O(1) (and nearly O(log n)) costs flat and multiply O(n) costs by about 10:

In [ ]:
def make_ops(n):
    """Operations on containers of size n: name -> (call, calls per timing run, claimed class)."""
    lst = list(range(n)); shuffled = random.sample(lst, n)
    st, dct, dq = set(lst), dict.fromkeys(lst), deque(lst)
    heap, text = list(range(n)), "a" * n            # a sorted list is already a valid heap
    return {
        "list[i]":                  (lambda: lst[n // 2], 2_000, "O(1)"),
        "list.append + pop()":      (lambda: (lst.append(0), lst.pop()), 2_000, "O(1)"),
        "x in list":                (lambda: -1 in lst, 1, "O(n)"),
        "list.insert(0) + pop(0)":  (lambda: (lst.insert(0, 0), lst.pop(0)), 10, "O(n)"),
        "x in set":                 (lambda: -1 in st, 2_000, "O(1)"),
        "dict[key]":                (lambda: dct[n // 2], 2_000, "O(1)"),
        "deque appendleft+popleft": (lambda: (dq.appendleft(0), dq.popleft()), 2_000, "O(1)"),
        "heappush + heappop":       (lambda: (heapq.heappush(heap, n // 2), heapq.heappop(heap)), 2_000, "O(log n)"),
        "sorted(random list)":      (lambda: sorted(shuffled), 1, "O(n log n)"),
        "'ab' in str":              (lambda: "ab" in text, 5, "O(n)"),
    }


ops_small, ops_big = make_ops(10_000), make_ops(100_000)
print(f"{'operation':26} {'claim':>10} {'n = 10k':>10} {'n = 100k':>10} {'ratio':>6}")
for name, (call_small, number, claim) in ops_small.items():
    t_small, t_big = interleaved_times([call_small, ops_big[name][0]], number=number, rounds=5)
    ratio = t_big / t_small
    print(f"{name:26} {claim:>10} {t_small * 1e6:>8.2f}µs {t_big * 1e6:>8.2f}µs {ratio:>6.1f}")
    assert (ratio < 2.5) if claim in ("O(1)", "O(log n)") else (ratio > 5), name

## 7. Hidden quadratics

Most accidental O(n²) in Python is an innocent-looking O(n) operation inside a loop:

| Looks harmless | Why it is O(n) per call | Fix |
|---|---|---|
| `if x not in seen_list:` | scans the list | `seen = set()` |
| `queue.pop(0)` | shifts every element left | `collections.deque().popleft()` |
| `s = s[1:]` or `a = a[1:]` | copies the rest | keep an index instead |
| `s += piece` in a loop | may copy the whole string each time | `"".join(pieces)` once |
| `sorted(...)` / `max(...)` inside a loop | O(n log n) / O(n) each time | sort once, or keep a heap |
| `bisect.insort(lst, x)` in a loop | the insert shifts elements | `heapq`, or a balanced tree / `sortedcontainers` |

The first one, measured (dedupe while keeping order — a real interview warm-up):

In [ ]:
def dedupe_list(items):
    seen, out = [], []
    for x in items:
        if x not in seen:          # scans `seen`: O(len(seen)) -> O(n²) overall
            seen.append(x)
            out.append(x)
    return out


def dedupe_set(items):
    seen, out = set(), []
    for x in items:
        if x not in seen:          # hash lookup: O(1) on average -> O(n) overall
            seen.add(x)
            out.append(x)
    return out


print(f"{'n':>6} {'list-based':>12} {'set-based':>11}")
for n in [250, 500, 1_000, 2_000]:
    items = list(range(n))                     # all distinct: the list version's worst case
    assert dedupe_list(items) == dedupe_set(items) == items
    t_list, t_set = interleaved_times([lambda: dedupe_list(items), lambda: dedupe_set(items)], rounds=2)
    print(f"{n:>6,} {t_list * 1e3:>10.2f}ms {t_set * 1e3:>9.3f}ms")
assert t_list > 20 * t_set
print("\ndoubling n: the list version grows ~4x per step, the set version ~2x")

## 8. Big-O in GenAI systems

The same reasoning answers three common GenAI interview questions.

### Attention is O(n²) in context length (`tr11`)

**In plain English:** in self-attention every token scores itself against every other token, so n tokens make n² scores. Double the context and you quadruple the work — and, if you store the score matrix, the memory.

Time the score matrix `Q @ Kᵀ` for one attention head (d = 64) as the context grows:

In [ ]:
from threadpoolctl import threadpool_limits    # installed with scikit-learn

d_head = 64
ctx_lengths = [256, 512, 1024, 2048]


def make_scores_call(n_tok):
    Q = rng.standard_normal((n_tok, d_head), dtype=np.float32)
    K = rng.standard_normal((n_tok, d_head), dtype=np.float32)
    return lambda: Q @ K.T                                      # an n x n matrix of scores


with threadpool_limits(limits=1):     # one BLAS thread, so thread start-up does not swamp the small sizes
    attn_times = interleaved_times([make_scores_call(n_tok) for n_tok in ctx_lengths])

attn_slope = np.polyfit(np.log2(ctx_lengths), np.log2(attn_times), 1)[0]
for n_tok, t in zip(ctx_lengths, attn_times):
    print(f"n = {n_tok:>5,} tokens: {n_tok * n_tok:>9,} scores in {t * 1e3:6.2f} ms")
print(f"fitted slope: {attn_slope:.2f}   (1 = linear, 2 = quadratic)")
assert attn_slope > 1.5

print("\nmemory of ONE n x n score matrix in fp16 (per head, per layer) if it is stored:")
for n_tok in [4_096, 32_768, 131_072]:
    print(f"  n = {n_tok:>7,}: {n_tok * n_tok * 2 / 1e9:7.2f} GB")

What people do about it: **FlashAttention** computes *exactly* the same result in tiles that fit in on-chip memory, so the n×n matrix is never written out — memory becomes O(n), compute stays O(n²), and it is much faster because it moves less data. **Sliding-window / sparse attention** (Mistral, Longformer) changes what each token can see (O(n·w) for a window w), and **linear attention / state-space models** (Mamba, RWKV) replace softmax attention with a recurrence that is O(n). The trap: FlashAttention is exact, the others are approximations of full attention.

### Exact vector search is O(N·d) per query (`ve03`)

Brute-force nearest-neighbour search scores the query against every stored vector: N dot products of length d. Measure it, then extrapolate to the interview's 10 million documents:

In [ ]:
dim, n_max, n_queries = 128, 32_000, 8
corpus = rng.standard_normal((n_max, dim), dtype=np.float32)
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)          # unit vectors: dot product = cosine
queries = corpus[:n_queries] + 0.05 * rng.standard_normal((n_queries, dim), dtype=np.float32)  # near-copies of docs 0-7


def exact_top_k(vectors, qs, k=5):
    """Top-k documents by cosine for a batch of queries: O(N·d) work per query."""
    scores = qs @ vectors.T                                   # (queries, N): every query vs every vector
    top = np.argpartition(-scores, k, axis=1)[:, :k]          # O(N) selection per query, no full sort
    order = np.argsort(-np.take_along_axis(scores, top, axis=1), axis=1)
    return np.take_along_axis(top, order, axis=1)


assert (exact_top_k(corpus, queries)[:, 0] == np.arange(n_queries)).all()   # each query finds its near-copy first
corpus_sizes = [4_000, 8_000, 16_000, 32_000]      # small enough to stay in CPU cache: clean timings
with threadpool_limits(limits=1):     # timed as a batch of 8 queries; per-query time = batch time / 8
    batch_times = interleaved_times([lambda N=N: exact_top_k(corpus[:N], queries) for N in corpus_sizes], number=5)
search_times = [t / n_queries for t in batch_times]
search_slope = np.polyfit(np.log2(corpus_sizes), np.log2(search_times), 1)[0]
for N, t in zip(corpus_sizes, search_times):
    print(f"N = {N:>7,} vectors x {dim} dims: {t * 1e3:6.2f} ms per query")
print(f"fitted slope: {search_slope:.2f}   (1 = linear)")
assert 0.6 < search_slope < 1.6

per_mult_add = search_times[-1] / (corpus_sizes[-1] * dim)
print(f"\nextrapolated to 10M docs x 768 dims: ~{per_mult_add * 10_000_000 * 768:.1f} s per query here,"
      f" scanning {10_000_000 * 768 * 4 / 1e9:.0f} GB of float32")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(ctx_lengths, np.array(attn_times) * 1e3, marker="o", color=COLORS[0])
axes[0].set(title=f"Attention scores: slope {attn_slope:.1f} (quadratic)",
            xlabel="context length n (tokens)", ylabel="time, ms (log scale)")
axes[1].loglog(corpus_sizes, np.array(search_times) * 1e3, marker="o", color=COLORS[1])
axes[1].set(title=f"Exact vector search: slope {search_slope:.1f} (linear)",
            xlabel="corpus size N (vectors)", ylabel="time per query, ms (log scale)")
for ax, sizes_used in zip(axes, [ctx_lengths, corpus_sizes]):
    ax.minorticks_off()
    ax.set_xticks(sizes_used, [f"{s:,}" for s in sizes_used])
plt.tight_layout()
plt.show()

So the answer to `ve03` starts with Big-O: exact search is linear in N — seconds per query on a laptop at 10M × 768, and tens to hundreds of milliseconds even on a big server or GPU — so production systems (1) **filter by metadata first** (tenant, date, language) to shrink N, then (2) use an **approximate** index — HNSW (greedy walk on a proximity graph) or IVF (search only the nearest clusters) — that inspects a tiny fraction of the vectors, trading a little recall for orders of magnitude of speed. See [09_rag/03 · Vector search](../09_rag/03_vector_search_faiss_hnsw.ipynb).

### The KV cache turns O(t²) per token into O(t) (`in02`)

**In plain English:** when generating token 500, the model needs keys and values for tokens 1–499. It already computed them while generating those tokens, and the causal mask means they never change — so it stores them (the KV cache) instead of recomputing. Counting the query·key dot products for one head in one layer:

In [ ]:
def attention_dot_products(n_tokens, kv_cache):
    """q·k products (one head, one layer) to generate n_tokens one at a time."""
    total = 0
    for t in range(1, n_tokens + 1):          # at step t the context holds t tokens
        # with a cache: one new query against t cached keys
        # without: recompute causal attention for all t positions
        total += t if kv_cache else t * (t + 1) // 2
    return total


print(f"{'tokens':>7} {'no cache':>16} {'with KV cache':>14} {'saving':>8}")
for n_tok in [256, 1_024, 4_096]:
    no_cache, cached = attention_dot_products(n_tok, False), attention_dot_products(n_tok, True)
    print(f"{n_tok:>7,} {no_cache:>16,} {cached:>14,} {no_cache / cached:>7.0f}x")
# 4x the tokens: ~64x the work without a cache (cubic in total), ~16x with one (quadratic in total)
assert 60 < attention_dot_products(4096, False) / attention_dot_products(1024, False) < 68
assert 15.9 < attention_dot_products(4096, True) / attention_dot_products(1024, True) < 16.1

What it **costs** is memory, and that is a space-complexity formula: 2 (K and V) × layers × KV heads × head dim × sequence length × batch × bytes per value — linear in sequence length and in batch size.

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch=1, bytes_per_value=2):
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bytes_per_value      # 2 = K and V


llama70b_like = dict(layers=80, kv_heads=8, head_dim=128)      # 70B-class model with grouped-query attention
print(f"per token (fp16): {kv_cache_bytes(**llama70b_like, seq_len=1) / 1024:.0f} KiB")
for seq_len in [8_192, 32_768, 131_072]:
    fp16 = kv_cache_bytes(**llama70b_like, seq_len=seq_len) / 1e9
    fp8 = kv_cache_bytes(**llama70b_like, seq_len=seq_len, bytes_per_value=1) / 1e9
    print(f"  one sequence of {seq_len:>7,} tokens: {fp16:6.1f} GB fp16, {fp8:6.1f} GB fp8")
print(f"  a batch of 32 at 8,192 tokens: {kv_cache_bytes(**llama70b_like, seq_len=8_192, batch=32) / 1e9:.0f} GB fp16")
assert kv_cache_bytes(**llama70b_like, seq_len=16_384) == 2 * kv_cache_bytes(**llama70b_like, seq_len=8_192)

That is why KV-cache size, not weight size, usually caps the batch size (and so the cost per token) of a serving deployment. The trap in `in02`: the KV cache does nothing for the *first* forward pass over the prompt (prefill), so it does not improve time-to-first-token — prompt caching is the different mechanism that does. More in [07_genai_foundations/06 · KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb).

## 9. How to state complexity in an interview

A four-part answer that works for every problem:

1. **Name the variables.** "Let n be the length of `nums` and k the window size."
2. **Time, with the reason.** Walk the code: sequential blocks *add*, nested loops *multiply* — unless the inner loop is bounded by something else. "Sorting is O(n log n); the two-pointer pass after it is O(n); total O(n log n)."
3. **Amortised arguments out loud.** "The `while` loop is nested, but every index is pushed and popped at most once, so all the pops together are O(n)."
4. **Space, and what you count.** "O(n) extra for the hash map; O(1) besides the output; the recursion adds O(depth)."

Then mention the case if it matters ("average O(1) — worst case O(n) with adversarial hash collisions") and connect to the constraints ("n ≤ 10⁵, so O(n log n) is comfortable and O(n²) would be 10¹⁰ steps").

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| n ≤ ~12 | O(n!) permutations / brute-force backtracking is intended |
| n ≤ ~25 | O(2ⁿ) subsets or bitmask DP |
| n ≤ a few hundred | O(n³) is fine (triple loop, interval DP) |
| n ≤ a few thousand | O(n²) is fine (all pairs, simple 2-D DP) |
| n ≈ 10⁵ – 10⁶ | O(n log n) or O(n): sort, heap, hash map, two pointers, sliding window |
| n or values up to 10⁹+ | O(log n) or O(1): binary search, math |
| a loop that halves / doubles the range | O(log n) |
| nested loops over the same data | O(n²) — can a hash map, a sort, or two pointers remove the inner loop? |
| a nested `while` that pops from a stack/window | often O(n) amortised — count total pushes and pops |
| `x in list`, `pop(0)`, slicing, `sorted()` inside a loop | a hidden O(n) factor |
| recursion with two branches and depth n | O(2ⁿ) — memoise (notebook 12) |
| "with O(1) extra space" | pointers / in-place writes, no hash set |

## Try it yourself

**1. Predict, then measure.** Write down the Big-O of each snippet before running the solution, which applies the doubling test to each one.

- (a) `sum(1 for x in a if x in b_list)` where `a` and `b_list` are both lists of length n
- (b) the same with `b_set`, a set of length n
- (c) `while n > 1: n //= 3` (count the steps)
- (d) `for i in range(n): for j in range(i, i + 5): total += j`

In [ ]:
# Solution — try it yourself first, then run this
def snippet_a(a, b_list):
    return sum(1 for x in a if x in b_list)


def snippet_b(a, b_set):
    return sum(1 for x in a if x in b_set)


def snippet_c(n):
    steps = 0
    while n > 1:
        n //= 3
        steps += 1
    return steps


def snippet_d(n):
    total = 0
    for i in range(n):
        for j in range(i, i + 5):          # the inner loop is always 5 steps, not n
            total += j
    return total


def growth_slope(make_call, n):
    """Fitted log-log slope over sizes n, 2n, 4n (the doubling test, three times)."""
    ns = [n, 2 * n, 4 * n]
    return np.polyfit(np.log2(ns), np.log2(interleaved_times([make_call(m) for m in ns])), 1)[0]


def make_a(n):
    a, b = list(range(n)), list(range(n, 2 * n))     # no overlap: every lookup scans all of b
    return lambda: snippet_a(a, b)


def make_b(n):
    a, b = list(range(n)), set(range(n, 2 * n))
    return lambda: snippet_b(a, b)


k_a, k_b = growth_slope(make_a, 500), growth_slope(make_b, 10_000)
k_d = growth_slope(lambda n: lambda: snippet_d(n), 10_000)
print(f"a: slope {k_a:.1f} -> O(n²)  (membership in a list inside a loop)")
print(f"b: slope {k_b:.1f} -> O(n)   (same code with a set)")
print(f"c: steps for n = 10**3, 10**6, 10**9: {[snippet_c(10 ** k) for k in (3, 6, 9)]} -> O(log n), base 3")
print(f"d: slope {k_d:.1f} -> O(n)   (5n steps: the inner loop has constant length)")
assert k_a > 1.5 > max(k_b, k_d)          # quadratic vs linear: split at the midpoint
assert all(abs(snippet_c(10 ** k) - math.log(10 ** k, 3)) <= 1 for k in (3, 6, 9))    # ≈ log₃ n steps

**2. Remove a hidden quadratic.** `strip_leading_slow` removes leading copies of a character by re-slicing the string — each slice copies the rest of the string. Make it O(n) (hint: keep an index), check it gives the same answers, and measure the speed-up.

In [ ]:
# Solution — try it yourself first, then run this
def strip_leading_slow(s, ch):
    while s and s[0] == ch:
        s = s[1:]                     # copies len(s) - 1 characters every step: O(n²) total
    return s


def strip_leading_fast(s, ch):
    i = 0
    while i < len(s) and s[i] == ch:  # just move an index: O(n) total
        i += 1
    return s[i:]                      # one copy at the end


for case in ["", "x", "xxx", "xxabc", "abc", "xaxb"]:
    assert strip_leading_fast(case, "x") == strip_leading_slow(case, "x") == case.lstrip("x")

big_text = "x" * 20_000 + "abc"
t_slow, t_fast, t_builtin = interleaved_times([lambda: strip_leading_slow(big_text, "x"),
                                               lambda: strip_leading_fast(big_text, "x"),
                                               lambda: big_text.lstrip("x")])
print(f"20,000 leading x's: slow {t_slow * 1e3:.1f} ms, fast {t_fast * 1e3:.2f} ms, "
      f"built-in lstrip {t_builtin * 1e6:.0f} µs (same O(n), but in C)")
assert t_builtin < t_fast < t_slow

**3. Why proportional growth?** Simulate three growth policies for a dynamic array — double the capacity, CPython's ~1.125×, and a fixed +16 slots — and count copies per append for n = 10³, 10⁴, 10⁵. Which policies give amortised O(1)?

In [ ]:
# Solution — try it yourself first, then run this
def simulated_copies_per_append(n, grow):
    capacity = size = copies = 0
    for _ in range(n):
        if size == capacity:
            copies += size              # move every element to the new block
            capacity = grow(capacity)
        size += 1
    return copies / n


policies = {"double (x2)":        lambda c: max(1, 2 * c),
            "CPython (~x1.125)":  lambda c: ((c + 1) + ((c + 1) >> 3) + 6) & ~3,
            "fixed +16 slots":    lambda c: c + 16}
table = {name: [simulated_copies_per_append(n, grow) for n in (1_000, 10_000, 100_000)]
         for name, grow in policies.items()}
print(f"{'policy':20} {'n=1e3':>8} {'n=1e4':>8} {'n=1e5':>8}   copies per append")
for name, row in table.items():
    print(f"{name:20} " + " ".join(f"{r:>8.1f}" for r in row))
assert max(table["double (x2)"]) < 2 and max(table["CPython (~x1.125)"]) < 10    # bounded: O(1) amortised
assert table["fixed +16 slots"][2] > 50 * table["fixed +16 slots"][0]            # grows with n: O(n) amortised

Geometric growth (any factor > 1) keeps copies per append bounded — amortised O(1). A fixed increment makes copies per append grow linearly with n, so n appends cost O(n²). A bigger factor means fewer copies but more wasted memory: CPython picks ~1.125 to save memory, doubling is the textbook choice.

## Say it in an interview

- **30-second answer:** "Big-O describes how time or memory grows with the input size, ignoring constant factors and lower-order terms. I give both time and space, in named variables, for the worst case unless I say otherwise, with the reason: O(n log n) because the sort dominates, O(n) extra for the hash map."
- **Key numbers:** log₂(10³) ≈ 10, log₂(10⁶) ≈ 20, log₂(10⁹) ≈ 30. Pure Python does roughly 10⁷ simple steps per second, so n = 10⁵ rules out O(n²). `list.append` is amortised O(1) because CPython over-allocates ~12.5%. Dict/set operations are O(1) on average, O(n) worst case. `sorted` is O(n log n), O(n) on already-sorted input.
- **Measure when unsure:** the doubling test — double n; ×1 constant/log, ×2 linear, ×4 quadratic.
- **Follow-ups they ask:** *"Is a dict lookup really O(1)?"* — on average; worst case O(n) with collisions, and hashing a string key costs O(length) (cached after the first time). *"What does amortised mean?"* — the average over a sequence of operations, guaranteed, not over random inputs. *"Why is `list.pop(0)` slow?"* — it shifts every element; use `deque`.
- **Traps:** nested loops are not automatically O(n²) (count total iterations: a monotonic stack is O(n)); O(1) does not mean fast (constants and cache misses); recursion costs O(depth) stack space and Python stops at ~1000 frames; slicing copies.
- **GenAI versions:** attention is O(n²) time in context length (FlashAttention makes memory O(n), exactly); exact vector search is O(N·d) per query, so filter first and use an ANN index; the KV cache trades O(layers × context × batch) memory for not recomputing the past.

## Next

- [02 · Arrays and two pointers](02_arrays_and_two_pointers.ipynb) — the first pattern that turns O(n²) into O(n)
- [04 · Stacks and queues](04_stacks_and_queues.ipynb) — `deque`, and why `list.pop(0)` is O(n)
- [06 · Binary search](06_binary_search.ipynb) — O(log n), and binary search on the answer
- [07 · Sorting algorithms](07_sorting_algorithms.ipynb) — O(n²) vs O(n log n) sorts, measured
- GenAI follow-ups: [Attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb), [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb), [Vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb)
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 1, "Big-O & complexity") · [interview bank](../../ai_interview_prep/index.html)